# Moment scaling analysis

The scaling behavior of statistical moments is investigated to detect  signatures of anomalous or strongly anomalous diffusion.

## 1. Imports and visualization settings

In [ ]:
from pathlib import Path
import pandas as pd
import logging
import numpy as np
import matplotlib.pyplot as plt
import pickle
from src import (
    integrate_to_velocity,
    integrate_to_displacement,
    analyze_all_windows,
    save_results_parquet,
    plot_scaling_curves,
    plot_scaling_exponents,
    set_plot_style
)
colors = set_plot_style()
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger()
def check(condition, message):
    if condition:
        logger.info(message)
    else:
        raise ValueError(message)
logger.info("Environment ready")

INFO | Environment ready


## 2. Data loading

In [ ]:
# Get project root
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

# Define all paths from project root
METADATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed' / '01_metadata'
SIGNALS_PROCESSED = PROJECT_ROOT / 'data' / 'processed' / '02_signals'
FIGURES_DIR = PROJECT_ROOT / 'figures' / '04_spatial' / '04b_moment_scaling'
LATEX_TABLES_DIR = PROJECT_ROOT / 'data' / 'processed' / 'latex_tables'

# Create output directories
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METADATA_PROCESSED.mkdir(parents=True, exist_ok=True)
SIGNALS_PROCESSED.mkdir(parents=True, exist_ok=True)
LATEX_TABLES_DIR.mkdir(parents=True, exist_ok=True)

check(FIGURES_DIR.exists(), f"Figures directory ready: {FIGURES_DIR}")
check(LATEX_TABLES_DIR.exists(), f"LaTeX tables directory ready: {LATEX_TABLES_DIR}")
check(METADATA_PROCESSED.exists(), f"Processed metadata directory ready: {METADATA_PROCESSED}")
check(SIGNALS_PROCESSED.exists(), f"Processed signals directory ready: {SIGNALS_PROCESSED}")

#Load metadata
logger.info("Loading metadata...")
df_meta = pd.read_parquet(METADATA_PROCESSED / 'metadata_clean.parquet')
check(df_meta is not None, "Metadata loaded successfully")
check(len(df_meta) > 0, "Metadata dataframe is not empty")
logger.info(f"Metadata loaded, shape: {df_meta.shape}")

# Load signals
logger.info("Loading acceleration signals...")
input_file = SIGNALS_PROCESSED / 'windowed_signals.pkl'
with open(input_file, 'rb') as f:
    windowed_signals = pickle.load(f)

## 4. Integration to velocity and displacement

Integrate acceleration once to obtain velocity, and twice to obtain displacement.
Baseline correction is crucial to prevent drift during integration.

In [4]:
logger.info("Integrating acceleration to velocity...")
df_vel = integrate_to_velocity(df_acc, method='trapz')
check(len(df_vel) == len(df_acc), "Velocity computed for all samples")

logger.info("Integrating acceleration to displacement...")
df_disp = integrate_to_displacement(df_acc, method='trapz')
check(len(df_disp) == len(df_acc), "Displacement computed for all samples")

logger.info("Integration complete.")

INFO | Integrating acceleration to velocity...
INFO | Velocity computed for all samples
INFO | Integrating acceleration to displacement...
INFO | Displacement computed for all samples
INFO | Integration complete.


## 5. Define analysis parameters

Set time lags (τ) and moment orders (q) for scaling analysis.

**Important**: τ values will be automatically filtered per window to fit the shortest window among all stations.

In [ ]:
# q values: moment orders from 0.5 to 5.0
q_values = np.arange(0.5, 5.1, 0.25)
logger.info(f"q values defined: {len(q_values)} values from {q_values.min()} to {q_values.max()}")

INFO | Tau values defined: 84 values from 1 to 10000 samples
INFO | q values defined: 19 values from 0.5 to 5.0


## 6. Windowed ensemble analysis

Compute increments, moments, and scaling exponents for all temporal windows and all three processes.

### Workflow:

1. **For each window** (pre-arrival, P-wave, S-wave, coda):
   - Find minimum window length across all stations
   - Filter τ values to fit shortest window
   - Compute increments with adaptive t₀ per station
   
2. **For each τ**:
   - Collect increments from all stations (ensemble)
   - Compute ensemble-averaged moments: $M_q(\tau) = \langle |\Delta x(\tau)|^q \rangle_{\text{stations}}$
   
3. **For each q**:
   - Fit scaling exponent: $M_q(\tau) \sim \tau^{\zeta(q)}$

This is done for acceleration, velocity, and displacement.

In [ ]:
results = analyze_all_windows(
    windowed_signals,
    tau_min=0.01,
    n_tau=None,  # automatico
    q_values=q_values,
    sampling_rate=200.0,
    fit_range=None
)

# Save results
save_results_parquet(results, output_dir='../data/processed/ensemble_spatial')

# Plot
plot_scaling_curves(results, output_dir= FIGURES_DIR / 'acceleration' / 'curves')
plot_scaling_exponents(results, output_dir= FIGURES_DIR / 'acceleration' / 'exponents')

INFO | 
################################################################################
INFO | # STARTING WINDOWED ENSEMBLE ANALYSIS
INFO | ################################################################################



################################################################################
# Processing: ACCELERATION
################################################################################

Computing increments for: pre_arrival

  Window 'pre_arrival' statistics:
    Valid files: 65
    Length range: [1480, 15452] samples
    Length range: [7.4, 77.3] s
    Mean length: 7833 samples (39.2 s)
    Filtered tau: removed 21/84 values (> 1479)
    Valid tau range: [1, 1417] samples
    Valid tau range: [0.005, 7.085] s

    Increments computed:
      Files: 65
      Tau values: 63
      Total increments: 4,095
      Increments per tau: 65

Computing increments for: p_wave

  Window 'p_wave' statistics:
    Valid files: 54
    Length range: [101, 13635] samples
    Length range: [0.5, 68.2] s
    Mean length: 5170 samples (25.9 s)
    Filtered tau: removed 50/84 values (> 100)
    Valid tau range: [1, 95] samples
    Valid tau range: [0.005, 0.475] s

    Increments computed:
      Files: 5

INFO | 
################################################################################
INFO | # ANALYSIS COMPLETE
INFO | ################################################################################



    Increments computed:
      Files: 48
      Tau values: 84
      Total increments: 4,032
      Increments per tau: 48

Computing moments for: pre_arrival

  Moment computation summary (ensemble):
    q values: 19
    Tau values: 63
    Stations per tau: 65
    Total moment values: 1197

Computing moments for: p_wave

  Moment computation summary (ensemble):
    q values: 19
    Tau values: 34
    Stations per tau: 54
    Total moment values: 646

Computing moments for: s_wave

  Moment computation summary (ensemble):
    q values: 19
    Tau values: 75
    Stations per tau: 66
    Total moment values: 1425

Computing moments for: coda

  Moment computation summary (ensemble):
    q values: 19
    Tau values: 84
    Stations per tau: 48
    Total moment values: 1596

Computing exponents for: pre_arrival

  Scaling exponents computed:
    q values: 19
    Mean R²: 0.9700
    Min R²: 0.9659

Computing exponents for: p_wave

  Scaling exponents computed:
    q values: 19
    Mean R²: 0

## 7. Results overview

Display scaling exponents ζ(q) for each window and process.

## 8. Visualization

### 8.1 Displacement: ζ(q) across windows

Compare scaling exponents for the four temporal windows.

**Expected behavior**:
- Pre-arrival: ζ(q) ≈ 0 (no scaling, noise)
- P-wave: ζ(q) > 0, linear with slope a
- S-wave: ζ(q) > 0, linear with slope b ≠ a (different regime)
- Coda: ζ(q) ≈ 0 (return to noise)

### 8.2 Displacement: Moments M_q(τ) for P-wave

Verify power-law scaling: $M_q(\tau) \sim \tau^{\zeta(q)}$

### 8.3 All processes: ζ(q) comparison for S-wave

Compare scaling behavior across acceleration, velocity, and displacement for S-wave window.

## 9. Summary

Create summary DataFrame with key results for each window and process.